# 08 Tag 批量管理

**用途**：查看/添加/清除 tag，安全开关（默认 ENABLE=False）防止误操作。


In [10]:
# ── 0. 配置区 ──────────────────────────────────────────────

DATASET_NAME  = "sahi_null_v2_ms1_0605-0621_40_ok"
TAG_TO_ADD    = "reviewed"     # 要批量添加的 tag
TAG_TO_CLEAR  = "reviewed"     # 要批量清除的 tag
MATCH_TAG     = ""             # 仅对已有此 tag 的样本操作（留空=全部样本）

ENABLE_ADD    = True          # 改为 True 才实际执行添加
ENABLE_CLEAR  = True          # 改为 True 才实际执行清除


In [2]:
# ── 1. 初始化日志 ──────────────────────────────────────────
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ 08_tag_batch_manager 初始化完成 ============")


2026-02-19 16:27:42,533 [INFO] ============ 08_tag_batch_manager 初始化完成 ============


## Step 1: 查看当前 Tag 分布


In [5]:
import fiftyone as fo
import pandas as pd
from IPython.display import display

logger.info("Step 1 开始：加载数据集并统计 tag 分布")

ds = fo.load_dataset(DATASET_NAME)
print(f"数据集: {ds.name}，样本数: {len(ds)}")

tag_counts = ds.count_values("tags")
if tag_counts:
    tag_df = pd.DataFrame(list(tag_counts.items()), columns=["tag", "count"]).sort_values("count", ascending=False)
    display(tag_df)
else:
    logger.info("（当前无 tags）")

logger.info(f"Step 1 完成：共 {len(tag_counts)} 种 tag")


2026-02-19 16:28:58,287 [INFO] Step 1 开始：加载数据集并统计 tag 分布
2026-02-19 16:28:58,292 [INFO] （当前无 tags）
2026-02-19 16:28:58,292 [INFO] Step 1 完成：共 0 种 tag


数据集: sahi_null_v2_ms1_0605-0621_40_ok，样本数: 243


## Step 2: 批量添加 Tag（ENABLE_ADD=True 才执行）


In [15]:
logger.info(f"Step 2 开始：ENABLE_ADD={ENABLE_ADD}")

if ENABLE_ADD:
    target_view = ds.match_tags(MATCH_TAG) if MATCH_TAG else ds
    target_view.tag_samples(TAG_TO_ADD)
    new_counts = ds.count_values("tags")
    logger.info(f"已添加 tag='{TAG_TO_ADD}'，当前分布: {new_counts}")
    logger.info(f"Step 2 完成：已给 {len(target_view)} 个样本添加 tag='{TAG_TO_ADD}'")
else:
    logger.info(f"ENABLE_ADD=False，跳过添加操作。目标 tag: '{TAG_TO_ADD}'")
    logger.info("Step 2 跳过：ENABLE_ADD=False")


2026-02-19 16:34:13,522 [INFO] Step 2 开始：ENABLE_ADD=True
2026-02-19 16:34:13,543 [INFO] 已添加 tag='reviewed'，当前分布: {'reviewed': 243}
2026-02-19 16:34:13,544 [INFO] Step 2 完成：已给 243 个样本添加 tag='reviewed'


## Step 3: 批量清除 Tag（ENABLE_CLEAR=True 才执行）


In [13]:
logger.info(f"Step 3 开始：ENABLE_CLEAR={ENABLE_CLEAR}")

if ENABLE_CLEAR:
    clear_view = ds.match_tags(TAG_TO_CLEAR)
    sample_count = len(clear_view)
    clear_view.untag_samples(TAG_TO_CLEAR)
    print(f"已清除 tag='{TAG_TO_CLEAR}'，共影响 {sample_count} 个样本")
    logger.info(f"Step 3 完成：清除了 {sample_count} 个样本的 tag='{TAG_TO_CLEAR}'")
else:
    print(f"ENABLE_CLEAR=False，跳过清除操作。目标 tag: '{TAG_TO_CLEAR}'")
    logger.info("Step 3 跳过：ENABLE_CLEAR=False")


2026-02-19 16:33:31,961 [INFO] Step 3 开始：ENABLE_CLEAR=True
2026-02-19 16:33:31,981 [INFO] Step 3 完成：清除了 243 个样本的 tag='reviewed'


已清除 tag='reviewed'，共影响 243 个样本


## Step 4: 导出各 Tag 的 filepath 列表


In [16]:
logger.info("Step 4 开始：导出各 tag 的 filepath 列表")

all_tags = list(ds.count_values("tags").keys())
for tag in all_tags:
    tag_view = ds.match_tags(tag)
    filepaths = tag_view.values("filepath")
    preview_df = pd.DataFrame({"filepath": filepaths[:5]})
    print(f"\nTag='{tag}'，共 {len(filepaths)} 个样本（前5条）:")
    display(preview_df)

logger.info("Step 4 完成")


2026-02-19 16:34:15,230 [INFO] Step 4 开始：导出各 tag 的 filepath 列表



Tag='reviewed'，共 243 个样本（前5条）:


,filepath
0,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...
1,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...
2,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...
3,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...
4,/home/tianqi/D/01_Projects/01_swd/02_code/pipe...


2026-02-19 16:34:15,237 [INFO] Step 4 完成


## 打开 App 按 Tag 筛选


In [ ]:
view_tag = TAG_TO_ADD if TAG_TO_ADD else None
if view_tag and view_tag in ds.count_values("tags"):
    app_view = ds.match_tags(view_tag)
    print(f"展示 tag='{view_tag}' 的样本")
else:
    app_view = ds
    print("展示全部样本")

session = fo.launch_app(app_view)
